In [5]:
!pip install azure-cognitiveservices-speech openai

  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
     ---------------------------------------- 0.0/109.4 kB ? eta -:--:--
     -------------------------------------- 109.4/109.4 kB 3.1 MB/s eta 0:00:00
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.3 MB 8.6 MB/s eta 0:00:01
   ------------------------------- -------- 1.1/1.3 MB 13.5 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 10.7 MB/s eta 0:00:00
Using cached anyio-4.13.0-p


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install azure-cognitiveservices-speech

  Using cached azure_cognitiveservices_speech-1.50.0-py3-none-win_amd64.whl.metadata (1.6 kB)
  Using cached azure_core-1.41.0-py3-none-any.whl.metadata (49 kB)
     ---------------------------------------- 0.0/41.7 kB ? eta -:--:--
     --------- ------------------------------ 10.2/41.7 kB ? eta -:--:--
     -------------------------------------  41.0/41.7 kB 653.6 kB/s eta 0:00:01
     -------------------------------------- 41.7/41.7 kB 403.8 kB/s eta 0:00:00
Using cached azure_cognitiveservices_speech-1.50.0-py3-none-win_amd64.whl (2.6 MB)
Using cached azure_core-1.41.0-py3-none-any.whl (220 kB)
   ---------------------------------------- 0.0/73.1 kB ? eta -:--:--
   ---------------------------------------- 73.1/73.1 kB 3.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/134.1 kB ? eta -:--:--
   ---------------------------------------- 134.1/134.1 kB 7.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/159.3 kB ? eta -:--:--
   ---------------------


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import azure.cognitiveservices.speech as speechsdk
from openai import AzureOpenAI

def run_dual_track_test(audio_file_path):
    print("🎙️ 투트랙 오디오 분석을 시작합니다...\n")
    
    # --------------------------------------------------
    # 트랙 1: Whisper (정확한 텍스트 및 혼용 언어 추출)
    # --------------------------------------------------
    openai_client = AzureOpenAI(
        api_key="4E9iixxpREIQezboFfIrh3SfWruRfs5nZAN6ijflKWlD0oWefzy3JQQJ99CEACI8hq2XJ3w3AAAAACOGDSDa",  
        api_version="2024-02-01", 
        azure_endpoint="https://9ai03-mpouyzd4-switzerlandnorth.services.ai.azure.com/" 
    )
    
    try:
        with open(audio_file_path, "rb") as audio_file:
            whisper_result = openai_client.audio.transcriptions.create(
                file=audio_file,
                model="drinkingmool-whisper" 
            )
        final_text = whisper_result.text
    except Exception as e:
        final_text = f"Whisper 에러 발생: {e}"

    # --------------------------------------------------
    # 트랙 2: Azure Speech (대본 없는 발음 평가 전용)
    # --------------------------------------------------
    speech_key = "7wPwCa2kS8FBb1bQZFEZTzQweikVVUifRzAStaIKRtkal2f4sEATJQQJ99CEACYeBjFXJ3w3AAAYACOG6gYf"
    service_region = "eastus" # 예: koreacentral
    
    speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=service_region)
    audio_config = speechsdk.AudioConfig(filename=audio_file_path) 
    
    # 프리토킹(대본 없음) 모드로 채점관 세팅
    pron_config = speechsdk.PronunciationAssessmentConfig(
        grading_system=speechsdk.PronunciationAssessmentGradingSystem.HundredMark,
        granularity=speechsdk.PronunciationAssessmentGranularity.Word
    )
    pron_config.reference_text = "" 
    
    recognizer = speechsdk.SpeechRecognizer(
        speech_config=speech_config, 
        audio_config=audio_config,
        language="en-US" 
    )
    pron_config.apply_to(recognizer)
    
    # 발음 채점 시작
    speech_result = recognizer.recognize_once_async().get()
    
    pron_score = 0
    if speech_result.reason == speechsdk.ResultReason.RecognizedSpeech:
        pronunciation_result = speechsdk.PronunciationAssessmentResult(speech_result)
        pron_score = pronunciation_result.pronunciation_score
    
    # --------------------------------------------------
    # 결과 브리핑
    # --------------------------------------------------
    print("=== 📊 [백엔드 최종 데이터 결과] ===")
    print(f"✅ 1. AI 캐릭터에게 전달할 텍스트: {final_text}")
    print(f"✅ 2. 시스템이 감지한 발음 점수: {pron_score} / 100")
    
    if pron_score < 50:
        print("\n🚨 [시스템 알림]: 발음 점수가 낮거나 한국어 혼용이 감지되었습니다!")
        print("   -> 프롬프트에 'is_penalty: true' 및 'affinity_delta: -3' 적용이 필요합니다.")
    else:
        print("\n✨ [시스템 알림]: 훌륭한 영어 발음입니다. 정상 대화를 진행합니다.")

# 코드 실행
run_dual_track_test("test2_audio.wav")

🎙️ 투트랙 오디오 분석을 시작합니다...

=== 📊 [백엔드 최종 데이터 결과] ===
✅ 1. AI 캐릭터에게 전달할 텍스트: 나는 완전 tired on today야
✅ 2. 시스템이 감지한 발음 점수: 91.2 / 100

✨ [시스템 알림]: 훌륭한 영어 발음입니다. 정상 대화를 진행합니다.
